# 02 — Procesamiento y Feature Engineering

**Objetivo:** Unificar los 6 módulos raw en un solo dataframe y construir las variables clínicas que alimentarán el modelo de supervivencia.

**Nodos replicados:** `merge_datasets` y `engineer_features` de `nodes.py`.

In [ ]:
%pip install pandas numpy pyarrow --quiet

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW   = Path('data/01_raw/nhanes_2013_2014')
INTER = Path('data/02_intermediate/nhanes_2013_2014')
PRIM  = Path('data/03_primary/nhanes_2013_2014')
for p in [INTER, PRIM]:
    p.mkdir(parents=True, exist_ok=True)
print('Rutas listas.')

Rutas listas.


## 1. Carga de datasets raw

Leemos los parquet generados en el notebook anterior (o por el pipeline Kedro).

In [2]:
demo      = pd.read_parquet(RAW / 'demo.parquet')
biopro    = pd.read_parquet(RAW / 'biopro.parquet')
bmx       = pd.read_parquet(RAW / 'bmx.parquet')
bpx       = pd.read_parquet(RAW / 'bpx.parquet')
smq       = pd.read_parquet(RAW / 'smq.parquet')
mortality = pd.read_parquet(RAW / 'mortality.parquet')

for name, df in [('demo',demo),('biopro',biopro),('bmx',bmx),
                  ('bpx',bpx),('smq',smq),('mortality',mortality)]:
    print(f'{name:10s}: {df.shape[0]:,} filas × {df.shape[1]} cols')

demo      : 10,175 filas × 47 cols
biopro    : 6,979 filas × 38 cols
bmx       : 9,813 filas × 26 cols
bpx       : 9,813 filas × 23 cols
smq       : 7,168 filas × 32 cols
mortality : 6,100 filas × 9 cols


## 2. Preparación de módulos (limpieza de SEQN)

`_prepare_module` hace 3 cosas por cada módulo:
1. Convierte `SEQN` a numérico (elimina ruido de lectura SAS)
2. Elimina duplicados por `SEQN` (cada participante aparece una sola vez)
3. Filtra solo las columnas que necesitamos (reduce memoria)

In [3]:
def _prepare_module(df, key='SEQN', keep=None):
    df = df.copy()
    df[key] = pd.to_numeric(df[key], errors='coerce').astype('Int64')
    df = df.dropna(subset=[key]).drop_duplicates(subset=[key])
    if keep:
        cols = [c for c in keep if c in df.columns]
        if key not in cols:
            cols = [key] + cols
        df = df[cols]
    return df.copy()

## 3. Merge de módulos clínicos

Usamos `SEQN` como llave de enlace. La estrategia de join es:
- `inner` con `biopro` y `mortality`: solo participantes con laboratorio y seguimiento completo
- `left` con `bmx`, `bpx`, `smq`: pueden tener datos faltantes (aceptamos NaN)

Esto replica exactamente el orden de merges de `merge_datasets()` en `nodes.py`.

In [4]:
demo_p   = _prepare_module(demo,   keep=['SEQN','RIDAGEYR','RIAGENDR'])
biopro_p = _prepare_module(biopro, keep=['SEQN','LBXSCR'])
bmx_p    = _prepare_module(bmx,    keep=['SEQN','BMXBMI'])
bpx_p    = _prepare_module(bpx,    keep=['SEQN','BPXSY1','BPXDI1'])
smq_p    = _prepare_module(smq,    keep=['SEQN','SMQ020'])
mort_p   = _prepare_module(mortality,
              keep=['SEQN','mortstat','permth_int','permth_exm','ucod_leading'])

merged = demo_p.merge(biopro_p, on='SEQN', how='inner')
merged = merged.merge(bmx_p,    on='SEQN', how='left')
merged = merged.merge(bpx_p,    on='SEQN', how='left')
merged = merged.merge(smq_p,    on='SEQN', how='left')
merged = merged.merge(mort_p,   on='SEQN', how='inner')
merged = merged.loc[:, ~merged.columns.duplicated()].copy()

print(f'Dataset merged: {merged.shape[0]:,} participantes × {merged.shape[1]} columnas')
print('Columnas:', merged.columns.tolist())

Dataset merged: 5,913 participantes × 12 columnas
Columnas: ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'LBXSCR', 'BMXBMI', 'BPXSY1', 'BPXDI1', 'SMQ020', 'mortstat', 'permth_int', 'permth_exm', 'ucod_leading']


In [5]:
merged.head(3)

,SEQN,RIDAGEYR,RIAGENDR,LBXSCR,BMXBMI,BPXSY1,BPXDI1,SMQ020,mortstat,permth_int,permth_exm,ucod_leading
0,73557,69.0,1.0,1.21,26.7,122.0,72.0,1.0,0.0,78.0,78.0,None
1,73558,54.0,1.0,0.79,28.6,156.0,62.0,1.0,1.0,58.0,58.0,007
2,73559,72.0,1.0,1.22,28.9,140.0,90.0,1.0,0.0,83.0,82.0,None


### Nulos por columna

In [6]:
nulos = merged.isnull().sum()
nulos[nulos > 0].sort_values(ascending=False)

ucod_leading    5469
BPXDI1           496
BPXSY1           496
LBXSCR           295
BMXBMI            77
dtype: int64

## 4. Feature Engineering

Construimos variables derivadas con mayor poder predictivo clínico:

| Variable | Descripción | Fórmula |
|---|---|---|
| `is_female` | Sexo binario | `RIAGENDR == 2` |
| `egfr` | Tasa filtración glomerular estimada (función renal) | Fórmula MDRD ajustada por sexo |
| `bmi` | Índice de masa corporal | `BMXBMI` |
| `map` | Presión arterial media | `DBP + (SBP - DBP) / 3` |
| `ever_smoked` | ¿Fumó alguna vez? | `SMQ020 == 1` |

La variable `egfr` es especialmente importante: la enfermedad renal crónica es un predictor robusto de mortalidad cardiovascular y por cualquier causa en adultos mayores.

In [7]:
df = merged.copy()

# is_female
df['is_female'] = pd.to_numeric(df['RIAGENDR'], errors='coerce').eq(2).astype('int8')

# eGFR (Fórmula MDRD — Modification of Diet in Renal Disease)
cr  = pd.to_numeric(df['LBXSCR'],   errors='coerce')
age = pd.to_numeric(df['RIDAGEYR'],  errors='coerce')
df['egfr'] = np.where(
    df['is_female'] == 1,
    186 * (cr ** -1.154) * (age ** -0.203) * 0.742,
    186 * (cr ** -1.154) * (age ** -0.203),
)

# BMI
df['bmi'] = pd.to_numeric(df['BMXBMI'], errors='coerce')

# MAP = DBP + (SBP - DBP) / 3
sbp = pd.to_numeric(df['BPXSY1'], errors='coerce')
dbp = pd.to_numeric(df['BPXDI1'], errors='coerce')
df['map'] = dbp + (sbp - dbp) / 3

# ever_smoked
df['ever_smoked'] = pd.to_numeric(df['SMQ020'], errors='coerce').eq(1).astype('int8')

### Selección final de columnas

Mantenemos solo las features que entran al modelo más las columnas de tiempo y evento.

In [8]:
DURATION_COL = 'permth_int'
EVENT_COL    = 'mortstat'

FEATURE_COLS = [
    'RIDAGEYR', 'RIAGENDR', 'LBXSCR',           # demografía + lab base
    'is_female', 'egfr', 'bmi', 'map',           # engineered
    'ever_smoked',                                # tabaco
    DURATION_COL, EVENT_COL,                      # target
]

features = df[[c for c in FEATURE_COLS if c in df.columns]].copy()

# Convertir a numérico y limpiar target
num_cols = [c for c in features.columns if c != EVENT_COL]
features[num_cols] = features[num_cols].apply(pd.to_numeric, errors='coerce')
features[EVENT_COL] = pd.to_numeric(features[EVENT_COL], errors='coerce')
features = features.dropna(subset=[DURATION_COL, EVENT_COL]).copy()
features[EVENT_COL] = features[EVENT_COL].astype('int8')

print(f'Features: {features.shape[0]:,} registros × {features.shape[1]} columnas')
print('Columnas:', features.columns.tolist())
print(f'Tasa de eventos: {features[EVENT_COL].mean()*100:.2f}%')

Features: 5,913 registros × 10 columnas
Columnas: ['RIDAGEYR', 'RIAGENDR', 'LBXSCR', 'is_female', 'egfr', 'bmi', 'map', 'ever_smoked', 'permth_int', 'mortstat']
Tasa de eventos: 7.51%


In [9]:
features.describe().round(2)

,RIDAGEYR,RIAGENDR,LBXSCR,is_female,egfr,bmi,map,ever_smoked,permth_int,mortstat
count,5913.00,5913.00,5618.00,5913.00,5618.00,5836.00,5417.00,5913.00,5913.00,5913.00
mean,47.41,1.52,0.91,0.52,92.25,28.92,87.11,0.42,70.40,0.08
std,18.43,0.50,0.52,0.50,26.75,7.18,12.07,0.49,12.25,0.26
min,18.00,1.00,0.30,0.00,2.18,14.10,32.67,0.00,2.00,0.00
25%,32.00,1.00,0.72,0.00,75.26,23.90,79.33,0.00,65.00,0.00
50%,47.00,2.00,0.85,1.00,90.89,27.70,86.00,0.00,72.00,0.00
75%,62.00,2.00,1.01,1.00,107.20,32.32,94.00,1.00,79.00,0.00
max,80.00,2.00,17.41,1.00,254.55,82.90,147.33,1.00,85.00,1.00


## 5. Persistencia

In [10]:
merged.to_parquet(INTER / 'merged.parquet', index=False)
features.to_parquet(PRIM / 'features.parquet', index=False)
print('✓ merged  →', INTER / 'merged.parquet')
print('✓ features →', PRIM / 'features.parquet')

✓ merged  → data\02_intermediate\nhanes_2013_2014\merged.parquet
✓ features → data\03_primary\nhanes_2013_2014\features.parquet
